# Face Detection CSV Deduplication

This notebook performs and documents the deduplication of the face-detection export used in this project.

Execution convention:

Run this notebook from its own directory, `code/scripts`. The repository's VS Code setting `jupyter.notebookFileRoot = ${fileDirname}` makes that the default in VS Code/Jupyter.

Input:

`../../data/datasets/TheEconomistHistoricalArchives-Faces.csv`

Outputs:

- `../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated.csv`
- `../../data/processed/TheEconomistHistoricalArchives-Faces-deduplication-audit.csv`

Deduplication rule:

A row is considered a clear duplicate when it has the same source scan/page stem and exactly the same relative bounding box as another row. Segmentation confidence, age estimate, gender score, literal filename suffix, and `Size relative` are retained as row attributes, but they do not define duplicate identity.

Representative-row rule:

When multiple rows share the same duplicate key, the retained row is selected deterministically by highest segmentation confidence, then by the cleanest `Size relative`/filename metadata, then by source order.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.options.display.max_columns = 60
pd.options.display.max_colwidth = 140

# Paths are relative to this notebook's directory: code/scripts.
raw_csv = Path("../../data/datasets/TheEconomistHistoricalArchives-Faces.csv")
deduplicated_csv = Path("../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated.csv")
audit_csv = Path("../../data/processed/TheEconomistHistoricalArchives-Faces-deduplication-audit.csv")

expected_columns = [
    "Filename",
    "Bounding Box relative X1",
    "Bounding Box relative Y1",
    "Bounding Box relative X2",
    "Bounding Box relative Y2",
    "Segmentation confidence score",
    "Size relative",
    "Age",
    "Gender",
]

bbox_columns = [
    "Bounding Box relative X1",
    "Bounding Box relative Y1",
    "Bounding Box relative X2",
    "Bounding Box relative Y2",
]
numeric_columns = bbox_columns + ["Segmentation confidence score", "Size relative", "Age", "Gender"]
duplicate_key = ["source_scan_id", *bbox_columns]

assert raw_csv.exists(), (
    f"Missing input file: {raw_csv}. "
    "Run this notebook from code/scripts. In VS Code, set jupyter.notebookFileRoot to ${fileDirname}."
)
deduplicated_csv.parent.mkdir(parents=True, exist_ok=True)

print(f"Input:  {raw_csv}")
print(f"Output: {deduplicated_csv}")
print(f"Audit:  {audit_csv}")


## Load the Source CSV

The source file is loaded with a fixed schema. A `source_row` column is added so every later output can be traced back to the original CSV row number.


In [ ]:
faces = pd.read_csv(raw_csv, dtype={"Filename": "string"})

assert list(faces.columns) == expected_columns, {
    "expected": expected_columns,
    "actual": list(faces.columns),
}
assert len(faces) > 0, "The input CSV is empty."

faces.insert(0, "source_row", np.arange(1, len(faces) + 1, dtype=np.int64))

for column in numeric_columns:
    faces[column] = pd.to_numeric(faces[column], errors="coerce")

print(f"Loaded {len(faces):,} face-detection rows.")
faces.head()


## Parse Filename Fields

The filename encodes the issue date and source page(s). For deduplication, the important derived field is `source_scan_id`: the filename stem before generated variant/crop suffixes.

Example: `1880-1120-0028_016_91.jpg` becomes `1880-1120-0028`.


In [ ]:
filename_parts = faces["Filename"].str.extract(
    r"^(?P<issue_id>\d{4}-\d{4})-(?P<source_pages>\d{4}(?:,\d{4})*)"
)

faces["source_scan_id"] = (
    faces["Filename"]
    .str.replace(r"\.[^.]+$", "", regex=True)
    .str.split("_", n=1)
    .str[0]
)
faces["issue_id"] = filename_parts["issue_id"]
faces["source_pages"] = filename_parts["source_pages"]
faces["issue_year"] = faces["issue_id"].str.slice(0, 4).astype("Int64")

parse_failures = int(faces["issue_id"].isna().sum())
assert parse_failures == 0, f"Could not parse {parse_failures:,} filenames."

bbox_width = faces["Bounding Box relative X2"] - faces["Bounding Box relative X1"]
bbox_height = faces["Bounding Box relative Y2"] - faces["Bounding Box relative Y1"]
faces["computed_bbox_area"] = bbox_width * bbox_height

pd.Series(
    {
        "rows": len(faces),
        "unique_literal_filenames": faces["Filename"].nunique(),
        "unique_source_scan_ids": faces["source_scan_id"].nunique(),
        "unique_issues": faces["issue_id"].nunique(),
        "year_min": int(faces["issue_year"].min()),
        "year_max": int(faces["issue_year"].max()),
    },
    name="value",
).to_frame()


## Sanity Checks

These checks document the state of the raw source file before deduplication. They are not used to remove rows.


In [ ]:
missing_values = (
    faces[expected_columns]
    .isna()
    .sum()
    .rename("missing_values")
    .to_frame()
)
missing_values


In [ ]:
numeric_summary = faces[numeric_columns].describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]).T
numeric_summary


In [ ]:
bbox_inside_unit_square = faces[bbox_columns].ge(0).all(axis=1) & faces[bbox_columns].le(1).all(axis=1)
size_area_difference = faces["Size relative"].sub(faces["computed_bbox_area"]).abs()

sanity_checks = pd.Series(
    {
        "bad_bbox_order": int(((bbox_width <= 0) | (bbox_height <= 0)).sum()),
        "bbox_coordinates_outside_0_1": int((~bbox_inside_unit_square).sum()),
        "missing_size_relative": int(faces["Size relative"].isna().sum()),
        "negative_size_relative": int((faces["Size relative"] < 0).sum()),
        "size_relative_gt_1": int((faces["Size relative"] > 1).sum()),
        "size_relative_differs_from_bbox_area": int((size_area_difference > 1e-8).sum()),
    },
    name="value",
)
sanity_checks.to_frame()


## Duplicate Definitions

The checks move from literal duplication to the project-specific duplicate key:

1. Exact full-row duplicates.
2. Same literal filename and exact bounding box.
3. Same source scan/page stem and exact bounding box.

Only the third definition is used for deduplication. It captures repeated detections on the same page(s), even if generated filename variants or model attributes differ.


In [ ]:
duplicate_definition_rows = []

for label, columns in [
    ("exact full source row", expected_columns),
    ("same literal filename and exact bbox", ["Filename", *bbox_columns]),
    ("same source scan/page stem and exact bbox", duplicate_key),
]:
    group_sizes = faces.groupby(columns, dropna=False, sort=False).size()
    duplicate_groups = group_sizes[group_sizes > 1]
    duplicate_definition_rows.append(
        {
            "definition": label,
            "duplicate_groups": len(duplicate_groups),
            "rows_in_duplicate_groups": int(duplicate_groups.sum()),
            "extra_rows_if_collapsed": int((duplicate_groups - 1).sum()),
            "largest_group": int(duplicate_groups.max()) if len(duplicate_groups) else 0,
        }
    )

duplicate_definition_summary = pd.DataFrame(duplicate_definition_rows)
duplicate_definition_summary


In [ ]:
clear_group_sizes = faces.groupby(duplicate_key, dropna=False, sort=False).size().rename("group_size")
clear_duplicate_groups = clear_group_sizes[clear_group_sizes > 1]
clear_duplicate_keys = clear_duplicate_groups.reset_index()[duplicate_key]
clear_duplicate_rows = faces.merge(clear_duplicate_keys, on=duplicate_key, how="inner")

model_variability_by_group = clear_duplicate_rows.groupby(duplicate_key, dropna=False).agg(
    distinct_confidence_scores=("Segmentation confidence score", "nunique"),
    distinct_age_estimates=("Age", "nunique"),
    distinct_gender_scores=("Gender", "nunique"),
)
model_variability_present = (
    (model_variability_by_group["distinct_confidence_scores"] > 1)
    | (model_variability_by_group["distinct_age_estimates"] > 1)
    | (model_variability_by_group["distinct_gender_scores"] > 1)
)

clear_duplicate_summary = pd.Series(
    {
        "clear_duplicate_groups": len(clear_duplicate_groups),
        "clear_duplicate_extra_rows": int((clear_duplicate_groups - 1).sum()),
        "rows_after_clear_deduplication": len(faces) - int((clear_duplicate_groups - 1).sum()),
        "duplicate_groups_with_different_model_outputs": int(model_variability_present.sum()),
    },
    name="value",
)
clear_duplicate_summary.to_frame()


In [ ]:
clear_duplicate_groups.value_counts().sort_index().rename_axis("group_size").rename("groups").reset_index()


## Choose the Retained Row

Duplicate identity is only page plus bounding box. Once duplicate groups are identified, the retained representative row is chosen by an explicit ranking:

1. Higher segmentation confidence.
2. Plausible `Size relative` in `[0, 1]`.
3. `Size relative` closest to the computed bounding-box area.
4. Higher filename quality score, when present in the filename.
5. Earlier filename variant index.
6. Earlier original source row.

The output keeps the original source columns from the selected representative row.


In [ ]:
filename_no_extension = faces["Filename"].str.replace(r"\.[^.]+$", "", regex=True)
filename_split = filename_no_extension.str.split("_", expand=True)

faces["filename_variant_index"] = pd.to_numeric(filename_split[1], errors="coerce").fillna(10**9).astype("int64")
faces["filename_quality_score"] = pd.to_numeric(filename_split[2], errors="coerce").fillna(-1).astype("int64")
faces["size_category"] = np.select(
    [
        faces["Size relative"].between(0, 1, inclusive="both"),
        faces["Size relative"] > 1,
        faces["Size relative"] < 0,
        faces["Size relative"].isna(),
    ],
    [0, 1, 2, 3],
    default=3,
)
faces["size_area_distance"] = faces["Size relative"].sub(faces["computed_bbox_area"]).abs().fillna(np.inf)

representatives = (
    faces.sort_values(
        [
            *duplicate_key,
            "Segmentation confidence score",
            "size_category",
            "size_area_distance",
            "filename_quality_score",
            "filename_variant_index",
            "source_row",
        ],
        ascending=[
            *([True] * len(duplicate_key)),
            False,
            True,
            True,
            False,
            True,
            True,
        ],
        kind="mergesort",
    )
    .drop_duplicates(subset=duplicate_key, keep="first")
    .sort_values("source_row", kind="mergesort")
    .copy()
)

assert representatives[duplicate_key].duplicated().sum() == 0
assert len(representatives) == clear_duplicate_summary.loc["rows_after_clear_deduplication"]

print(f"Selected {len(representatives):,} retained detections.")


In [ ]:
sample_duplicate_keys = clear_duplicate_keys.head(10)

sample_duplicate_rows = (
    faces.merge(sample_duplicate_keys, on=duplicate_key, how="inner")
    .sort_values(["source_scan_id", *bbox_columns, "source_row"], kind="mergesort")
)

sample_duplicate_rows[
    [
        "source_scan_id",
        "source_row",
        "Filename",
        "Segmentation confidence score",
        "Age",
        "Gender",
        "Size relative",
        "computed_bbox_area",
        "filename_variant_index",
        "filename_quality_score",
    ]
].head(80)


## Write the Deduplicated CSV and Audit CSV

The deduplicated CSV contains only the original source columns. The audit CSV records every row from every collapsed duplicate group and marks the retained row with `action == "keep"`.


In [ ]:
selected_source_rows = set(representatives["source_row"])

duplicate_group_ids = clear_duplicate_keys.copy()
duplicate_group_ids.insert(0, "duplicate_group_id", np.arange(1, len(duplicate_group_ids) + 1, dtype=np.int64))

audit_rows = (
    faces.merge(duplicate_group_ids, on=duplicate_key, how="inner")
    .sort_values(["duplicate_group_id", "source_row"], kind="mergesort")
    .copy()
)
audit_rows["action"] = np.where(audit_rows["source_row"].isin(selected_source_rows), "keep", "remove")

selected_lookup = representatives[[*duplicate_key, "source_row", "Filename"]].rename(
    columns={"source_row": "selected_source_row", "Filename": "selected_filename"}
)
audit_rows = audit_rows.merge(selected_lookup, on=duplicate_key, how="left")

kept_per_group = audit_rows.groupby("duplicate_group_id")["action"].apply(lambda actions: (actions == "keep").sum())
assert audit_rows["selected_source_row"].notna().all()
assert (kept_per_group == 1).all()

deduplicated_output = representatives[expected_columns].copy()
audit_output = audit_rows[
    [
        "duplicate_group_id",
        "action",
        "source_row",
        "selected_source_row",
        "source_scan_id",
        "selected_filename",
        *expected_columns,
    ]
].copy()

deduplicated_output.to_csv(deduplicated_csv, index=False)
audit_output.to_csv(audit_csv, index=False)

pd.Series(
    {
        "deduplicated_rows_written": len(deduplicated_output),
        "audit_rows_written": len(audit_output),
        "deduplicated_csv": str(deduplicated_csv),
        "audit_csv": str(audit_csv),
    },
    name="value",
).to_frame()


## Verify the Written Files

The output files are reloaded from disk and checked against the intended duplicate key. This catches accidental write-path or serialization mistakes.


In [ ]:
deduplicated_check = pd.read_csv(deduplicated_csv, dtype={"Filename": "string"})
audit_check = pd.read_csv(audit_csv, dtype={"Filename": "string", "selected_filename": "string"})

assert list(deduplicated_check.columns) == expected_columns
assert len(deduplicated_check) == len(deduplicated_output)
assert len(audit_check) == len(audit_output)

check = deduplicated_check.copy()
check["source_scan_id"] = (
    check["Filename"]
    .str.replace(r"\.[^.]+$", "", regex=True)
    .str.split("_", n=1)
    .str[0]
)
for column in bbox_columns:
    check[column] = pd.to_numeric(check[column], errors="coerce")

remaining_duplicate_rows = int(check.duplicated(subset=duplicate_key, keep=False).sum())
assert remaining_duplicate_rows == 0, f"Found {remaining_duplicate_rows:,} remaining clear duplicate rows."

pd.Series(
    {
        "deduplicated_rows": len(deduplicated_check),
        "audit_rows": len(audit_check),
        "remaining_page_bbox_duplicate_rows": remaining_duplicate_rows,
    },
    name="value",
).to_frame()


## IoU Diagnostic for Likely Near-Duplicates

IoU means Intersection over Union: `area(box overlap) / area(area covered by either box)`. It ranges from 0.0 for no overlap to 1.0 for identical boxes.

The deduplication above removes exact same-page/same-box repeats. This diagnostic asks a stricter follow-up question: after that cleanup, are there still retained detections on the same source scan/page whose boxes almost overlap? If no pairs reach even IoU >= 0.80, there is no evidence for another automatic deduplication pass. Manual review heuristics would usually use a stricter threshold such as IoU >= 0.95.


In [ ]:
iou_thresholds = [0.80, 0.90, 0.95, 0.98, 0.99]
threshold_counts = {threshold: 0 for threshold in iou_thresholds}
near_duplicate_examples = []
max_iou_observed = 0.0
pair_comparisons = 0
multi_face_source_scans = 0

for source_scan_id, group in representatives.groupby("source_scan_id", sort=False):
    if len(group) < 2:
        continue

    multi_face_source_scans += 1
    boxes = group[bbox_columns].to_numpy(dtype=float)
    rows = group["source_row"].to_numpy()
    filenames = group["Filename"].astype(str).to_numpy()

    for i in range(len(group)):
        x1 = np.maximum(boxes[i, 0], boxes[i + 1 :, 0])
        y1 = np.maximum(boxes[i, 1], boxes[i + 1 :, 1])
        x2 = np.minimum(boxes[i, 2], boxes[i + 1 :, 2])
        y2 = np.minimum(boxes[i, 3], boxes[i + 1 :, 3])

        intersection = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
        area_i = (boxes[i, 2] - boxes[i, 0]) * (boxes[i, 3] - boxes[i, 1])
        area_j = (boxes[i + 1 :, 2] - boxes[i + 1 :, 0]) * (boxes[i + 1 :, 3] - boxes[i + 1 :, 1])
        union = area_i + area_j - intersection
        iou = np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)

        pair_comparisons += len(iou)
        if len(iou):
            max_iou_observed = max(max_iou_observed, float(iou.max()))

        for threshold in iou_thresholds:
            matches = np.flatnonzero(iou >= threshold)
            threshold_counts[threshold] += len(matches)

            for position in matches[: max(0, 20 - len(near_duplicate_examples))]:
                j = i + 1 + position
                near_duplicate_examples.append(
                    {
                        "threshold": threshold,
                        "source_scan_id": source_scan_id,
                        "left_source_row": int(rows[i]),
                        "right_source_row": int(rows[j]),
                        "iou": float(iou[position]),
                        "left_filename": filenames[i],
                        "right_filename": filenames[j],
                    }
                )

print(f"Retained detections checked: {len(representatives):,}")
print(f"Source scan/page stems with multiple retained detections: {multi_face_source_scans:,}")
print(f"Same-source retained detection pairs compared: {pair_comparisons:,}")
print(f"Maximum IoU observed after clear deduplication: {max_iou_observed:.3f}")

iou_summary = pd.DataFrame(
    [{"iou_threshold": threshold, "same_source_pairs": count} for threshold, count in threshold_counts.items()]
)
display(iou_summary)

near_duplicate_examples = pd.DataFrame(near_duplicate_examples)
if near_duplicate_examples.empty:
    print(
        "No same-source retained detection pairs reach IoU >= 0.80. "
        "Since even this loose review threshold finds zero candidates, "
        "there is no evidence that another automatic deduplication pass is needed."
    )
else:
    print("Potential near-duplicate candidates for manual review:")
    display(near_duplicate_examples.sort_values("iou", ascending=False).head(20))


## Conclusion

This notebook performs one conservative, reproducible cleanup: exact same-page/same-box duplicate detections are collapsed to one retained row, with a full audit trail. The IoU diagnostic then checks whether another near-duplicate pass appears warranted. For the current extract, the diagnostic output shows no high-overlap retained pairs at IoU >= 0.80, so further automatic deduplication is not supported by the data.
